# Medical LoRA Fine-tuning — TechCorp Hackathon

**Model**: `microsoft/Phi-3.5-mini-instruct`  
**Dataset**: `ruslanmv/ai-medical-chatbot`  
**Technique**: QLoRA (4-bit quantization + LoRA adapters)

> **Runtime required**: GPU (T4 minimum) — Runtime > Change runtime type > T4 GPU

---

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q transformers==4.45.0
!pip install -q peft==0.12.0
!pip install -q accelerate==0.34.0
!pip install -q bitsandbytes==0.43.1
!pip install -q datasets==2.20.0
!pip install -q trl==0.11.0
!pip install -q evaluate
print('Dependencies installed.')

## 2. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} — {mem:.1f} GB VRAM')
else:
    print('No GPU detected. Switch runtime to GPU before continuing.')

## 3. Load and inspect the medical dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('ruslanmv/ai-medical-chatbot', split='train')
print(f'Total entries: {len(dataset)}')
print('Columns:', dataset.column_names)
print('\nSample entry:')
print(dataset[0])

## 4. Preprocess — format for instruction fine-tuning

In [ ]:
import re

# Phi-3.5 chat template
def format_example(example):
    patient = (example.get('Patient') or example.get('question') or '').strip()
    doctor  = (example.get('Doctor')  or example.get('answer')   or '').strip()

    if not patient or not doctor:
        return {'text': None}
    if len(patient) < 20 or len(doctor) < 20:
        return {'text': None}

    text = (
        f'<|user|>\n{patient}<|end|>\n'
        f'<|assistant|>\n{doctor}<|end|>'
    )
    return {'text': text}

dataset = dataset.map(format_example)
dataset = dataset.filter(lambda x: x['text'] is not None)

# Train / validation split
split = dataset.train_test_split(test_size=0.05, seed=42)
train_data = split['train']
eval_data  = split['test']

print(f'Train: {len(train_data)} | Eval: {len(eval_data)}')
print('\nFormatted example:')
print(train_data[0]['text'][:300])

## 5. Load tokenizer and base model (4-bit QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'microsoft/Phi-3.5-mini-instruct'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
print('Base model loaded.')

## 6. Apply LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Tokenize dataset

In [ ]:
MAX_LENGTH = 512

def tokenize(examples):
    out = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )
    out['labels'] = out['input_ids'].copy()
    return out

tokenized_train = train_data.map(tokenize, batched=True, remove_columns=train_data.column_names)
tokenized_eval  = eval_data.map(tokenize,  batched=True, remove_columns=eval_data.column_names)

print(f'Tokenized train: {len(tokenized_train)} | eval: {len(tokenized_eval)}')

## 8. Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = './phi35_medical_lora'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    remove_unused_columns=False,
    dataloader_drop_last=True,
    report_to='none',
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print('Starting training...')
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)

print('\nTraining stats:')
print(f'  Final loss  : {train_result.training_loss:.4f}')
print(f'  Total steps : {train_result.global_step}')
print(f'  Runtime (s) : {train_result.metrics["train_runtime"]:.1f}')

## 9. Evaluation — loss on validation set

In [ ]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])

print('Evaluation results:')
for k, v in eval_results.items():
    print(f'  {k}: {v:.4f}')
print(f'  Perplexity: {perplexity:.2f}')

## 10. Test — inference on sample medical questions

In [ ]:
model.eval()

TEST_QUESTIONS = [
    'I have a persistent headache for 3 days and a mild fever. What could it be?',
    'What are the common side effects of ibuprofen?',
    'How long does a typical flu last?',
]

def generate(question: str, max_new_tokens: int = 200) -> str:
    prompt = f'<|user|>\n{question}<|end|>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.replace('<|end|>', '').strip()

print('=== Medical model inference test ===')
for q in TEST_QUESTIONS:
    print(f'\nQ: {q}')
    print(f'A: {generate(q)}')
    print('-' * 60)

## 11. Save LoRA adapters to Google Drive (optional)

In [ ]:
# Uncomment to mount Google Drive and save the adapters
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(OUTPUT_DIR, '/content/drive/MyDrive/phi35_medical_lora')
# print('Adapters saved to Google Drive.')

print(f'Adapters are in: {OUTPUT_DIR}')
!ls -lh {OUTPUT_DIR}

## 12. Summary — metrics to report

Fill in after training:

| Metric | Value |
|--------|-------|
| Model | Phi-3.5-mini-instruct |
| Dataset | ruslanmv/ai-medical-chatbot |
| Technique | QLoRA (4-bit, r=16) |
| Epochs | 3 |
| Final train loss | — |
| Eval loss | — |
| Perplexity | — |
| Training time | — |

> **Disclaimer**: This model is experimental and for research purposes only.
> It must NOT be used for actual medical decisions.
> Always consult a qualified healthcare professional.